<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Count and detect in crowded scenes

**[Count and detect in crowded scenes](https://learning.nextia-ai.com/courses/vision/m09/count-and-detect-in-crowded-scenes/)** · Vision AI: See, Count and Inspect · Case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** count windows and cars in real open photos, from a few objects to crowded scenes. You compare five counters on the same photos and under one measure: the mean count (a baseline), RT-DETR trained on COCO with no new training, Faster R-CNN fine-tuned for a few CPU minutes, a density estimate, and the recorded answers of a hosted vision-language model. You measure the count error by crowding, choose a line above which a person counts the photo, test once, and run the decision on new photos.

| | |
|---|---|
| **Time** | About 2 hours and 30 minutes with the lesson page. The notebook runs in about 4 to 5 minutes on a laptop CPU (the fine-tuning takes about 173 seconds), plus the downloads (the photos 15 MB, RT-DETR 81 MB, the detector 74 MB, ResNet-18 47 MB). |
| **Needs** | An internet connection for the setup cells and the models. No account and no key: the hosted model's answers are recorded. |
| **You should know** | Module 4 of this course: [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/), [Pretrained detectors](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/), [Fine-tune a detector on a few images](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/) and [Counting, and tracking in brief](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1, matplotlib 3.11.2, PyTorch 2.15.0, torchvision 0.29.2, timm 1.0.31 and Transformers 5.22.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab, Kaggle, Windows and Linux are not tested |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m09/count-and-detect-in-crowded-scenes/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M09-L03-count-and-detect-in-crowded-scenes/count-and-detect-in-crowded-scenes-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, transformers 5.22.2, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 transformers==5.22.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import transformers; print("transformers", transformers.__version__)

## Setup: get the photos

Run the next cell. It downloads a fixed sample of Open Images photos with their boxes (about 14 MB, one zip file) and the course's helper `compare.py` into a folder `crowded-scenes`. It checks each file's checksum and unpacks the zip. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `crowded-scenes` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: [Open Images V7](https://storage.googleapis.com/openimages/web/index.html). Each photo is by its Flickr author under [CC BY 2.0](https://creativecommons.org/licenses/by/2.0/) (the author and the Flickr page of every photo are in `data/crowd/manifest.csv`); the boxes are by Google under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning chose 198 photos with no person in them, resized them and converted the boxes to pixels. The file `dataset.md` next to this notebook lists every change.

In [ ]:
# Setup: download the photos and compare.py into crowded-scenes/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "crowd.zip": "0cf4aad01fc379c0e1d7c86eb169b6f5db013503585590f48a8b4e87c24fae68",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("crowded-scenes").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + ("M09-L03-count-and-detect-in-crowded-scenes/" if name.endswith(".zip") else "") + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder crowded-scenes and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in crowded-scenes/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation and the cost table

Clearline (a made-up company) cleans windows and washes cars at offices. A customer sends a photo, and Clearline quotes a price from a count: the windows of the building, or the cars in the car park. Every item counted wrong costs money: too few, and Clearline does work it is not paid for; too many, and the customer finds the quote too high. The office can count some photos by hand. These are **assumptions**, not measured facts:

- a window counted wrong costs 3 dollars (the price of cleaning one window);
- a car counted wrong costs 12 dollars (the price of washing one car);
- a person counts one photo for 0.50 dollars (about 1 minute at 30 dollars an hour), and is always right;
- the office can count at most 250 photos in every 1,000.

## 2. The data and the first checks

Each row of the manifest is one photo. The columns that matter: `class` (what is counted: `window` or `car`), `count` (the number of boxes that Open Images' annotators drew: the true count), `band` (the crowding band by that count: 3-5, 6-9 or 10+) and `split`. Each row of `boxes.csv` is one true box, in pixels.

In [ ]:
photos = pd.read_csv("data/crowd/manifest.csv")
boxes = pd.read_csv("data/crowd/boxes.csv")
CLASSES, BANDS = ["window", "car"], ["3-5", "6-9", "10+"]
print(photos.pivot_table(index=["class", "split"], columns="band", values="photo_id", aggfunc="count", fill_value=0))

> **Check.** You should see 39 + 28 training photos, 24 + 19 validation photos, 36 + 31 test photos and 12 + 9 new photos (windows + cars). Only 15 car photos in the sample have 10 cars or more: crowded car photos without a person and without a "group" box are rare in Open Images.

Three checks: no photo is in the sample twice, every count matches its boxes, and how large the boxes are. `side` is the side of a square with the same area as the box, in pixels of a photo 640 pixels wide.

In [ ]:
print("photos:", len(photos), "| different source photos:", photos["source_sha256"].nunique())
print("boxes:", len(boxes), "| counts match the boxes:",
      (boxes.groupby("photo_id").size().reindex(photos["photo_id"]).values == photos["count"].values).all())
boxes["side"] = np.sqrt(boxes["w"] * boxes["h"])  # the side of a square with the same area, in pixels
print(boxes.groupby("class")["side"].describe(percentiles=[0.1, 0.5]).round(1)[["count", "10%", "50%", "max"]])
print(photos.groupby("class")["count"].describe()[["min", "50%", "max"]])

Now look at some photos with their true boxes: one window photo and one car photo of each band, from the training split.

In [ ]:
import matplotlib.patches as patches
pick = [photos[(photos["class"] == c) & (photos["split"] == "train") & (photos["band"] == b)].iloc[0]
        for c in CLASSES for b in BANDS]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
for ax, r in zip(axes.ravel(), pick):
    ax.imshow(Image.open("data/crowd/" + r["file"]))
    for x, y, w, h in boxes[boxes["photo_id"] == r["photo_id"]][["x", "y", "w", "h"]].values:
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="orangered", lw=1))
    ax.set_title(f"{r['class']}: {r['count']}  ({r['author'][:24]}, CC BY 2.0)", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

> **Predict.** Look at the window photos. Would you count the same number of windows as the boxes? Is a row of glass panes one window or several?

## 3. The cost table and the scores

`count_rows` gives the comparison-table rows of one counter (the count error by class, with the worst crowding band), as `compare.count_row` does in Module 4. `quote_cost` turns counts into a cost per 1,000 photos: photos at or above a **line** go to a person while the budget lasts, and the others are quoted from the count. `choose_line` finds the cheapest line on the **validation** photos.

In [ ]:
# Clearline's cost table (assumptions, not measured).
COSTS = {
    "window": 3.0,               # dollars lost for each window counted wrong: the price of cleaning one window
    "car": 12.0,                 # dollars lost for each car counted wrong: the price of washing one car
    "check_by_a_person": 0.5,    # a person counts one photo: about 1 minute at 30 dollars an hour
    "checks_per_1000": 250,      # the office can count at most 250 photos in every 1,000 by hand
}
TRUE = dict(zip(photos["photo_id"], photos["count"]))
def split_photos(cls, split):
    return photos[(photos["class"] == cls) & (photos["split"] == split)]

def count_rows(method, counts, split="test"):
    """One comparison-table row per class: the count error (MAE) on the split, worst crowding band."""
    rows = []
    for cls in CLASSES:
        p = split_photos(cls, split)
        rows.append(cp.count_row(method, p["count"].tolist(), [counts[i] for i in p["photo_id"]],
                                 p["band"].tolist(), f"Open Images {cls}s {split}", None, None, None, None)
                    | {"class": cls})
    return rows

def quote_cost(counts, cls, split, line=float("inf"), costs=None):
    """Cost per 1,000 photos. A photo whose count is at or above the line goes to a person (exact, 0.50 dollars),
    while the budget lasts: when more photos cross the line, the person takes those with the highest counts.
    Every other photo is quoted from its count, and each item counted wrong costs the class's price.
    Only photos with a count are used (the recorded model has no answer for some validation photos)."""
    costs = costs or COSTS
    p = split_photos(cls, split)
    p = p[p["photo_id"].isin(list(counts))]
    c = np.array([counts[i] for i in p["photo_id"]], float)
    room = int(len(c) * costs["checks_per_1000"] / 1000)
    order = np.lexsort((p["photo_id"].values, -c))  # highest count first; ties in the order of the photo IDs (hashes)
    to_person = np.zeros(len(c), bool)
    to_person[order[:room]] = True
    to_person &= np.round(c) >= line  # the line is a whole number of items
    cost = np.where(to_person, costs["check_by_a_person"], np.abs(np.round(c) - p["count"].values) * costs[cls])
    return round(float(cost.mean() * 1000), 1), round(float(to_person.mean() * 1000), 1)

def choose_line(counts, cls, costs=None):
    """The cheapest 'send to a person from this count' line on the validation photos (inf: never),
    among the lines that send at most the budget of photos to a person."""
    costs = costs or COSTS
    p = split_photos(cls, "val")
    p = p[p["photo_id"].isin(list(counts))]
    c = np.array([counts[i] for i in p["photo_id"]], float)
    options = sorted({float(np.round(v)) for v in c}) + [float("inf")]
    return min(options, key=lambda t: (quote_cost(counts, cls, "val", t, costs)[0], -t))

## 4. A baseline: the mean count

The first counter does not look at the photo. It gives every photo the mean count of the training photos of its class. Any real counter must beat it.

In [ ]:
# Baseline: no look at the photo. Every photo gets the mean count of the training photos of its class.
mean_count = photos[photos["split"] == "train"].groupby("class")["count"].mean()
baseline = {i: float(mean_count[c]) for i, c in zip(photos["photo_id"], photos["class"])}
print(mean_count.round(1).to_dict())

## 5. Detector 1: RT-DETR, trained on COCO, no new training

COCO, the data that most public detectors learn from, has a class "car". It has no class "window". This cell downloads RT-DETR R18 (81 MB) and runs it on the 131 validation, test and new photos, in about 20 seconds.

In [ ]:
import time, torch
from transformers import AutoImageProcessor, RTDetrForObjectDetection
torch.manual_seed(0)
torch.set_num_threads(4)
# RT-DETR R18 trained on COCO (Apache-2.0), at a fixed revision, as in Module 4. COCO has "car" but no "window".
REV = "ac77a11ff0170a41b771c03264987f8ce2b0d753"
processor = AutoImageProcessor.from_pretrained("PekingU/rtdetr_r18vd", revision=REV)
rtdetr = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r18vd", revision=REV).eval()
COCO_NAME = {"car": "car", "window": None}

RT_BOXES = {}  # photo -> [(score, [x, y, w, h])] of the photo's class, score >= 0.05
start = time.perf_counter()
for i, f, cls in photos[photos["split"] != "train"][["photo_id", "file", "class"]].itertuples(index=False):
    img = Image.open("data/crowd/" + f).convert("RGB")
    with torch.no_grad():
        out = rtdetr(**processor(images=img, return_tensors="pt"))
    r = processor.post_process_object_detection(out, target_sizes=torch.tensor([[img.height, img.width]]),
                                                threshold=0.05)[0]
    RT_BOXES[i] = [(s, [b[0], b[1], b[2] - b[0], b[3] - b[1]])
                   for b, s, l in zip(r["boxes"].tolist(), r["scores"].tolist(), r["labels"].tolist())
                   if rtdetr.config.id2label[l] == COCO_NAME[cls]]
rt_ms = 1000 * (time.perf_counter() - start) / len(RT_BOXES)
print(f"RT-DETR: {len(RT_BOXES)} photos, {rt_ms:.0f} ms a photo")

The count of a detector depends on its **score line**: only boxes with a score at or above it count. We choose one line per class on the validation photos, as in [Fine-tune a detector on a few images](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/).

In [ ]:
def counts_at(found, line):
    """The count of each photo: its boxes with a score at or above the line."""
    return {i: sum(1 for s, _ in b if s >= line) for i, b in found.items()}

def choose_score_line(found, cls):
    """The score line with the lowest count error on the validation photos of one class."""
    p = split_photos(cls, "val")
    def err(t):
        return np.mean([abs(sum(1 for s, _ in found[i] if s >= t) - n) for i, n in zip(p["photo_id"], p["count"])])
    return min(np.round(np.arange(0.05, 0.96, 0.05), 2), key=lambda t: (err(t), t))

def detector_counts(found):
    """Counts with a score line per class, chosen on validation."""
    lines = {cls: float(choose_score_line(found, cls)) for cls in CLASSES}
    counts = {}
    for cls in CLASSES:
        ids = photos[(photos["class"] == cls) & (photos["split"] != "train")]["photo_id"]
        counts.update(counts_at({i: found[i] for i in ids}, lines[cls]))
    return counts, lines

def ap50(found, cls, split="test"):
    """AP at IoU 0.5 for one class on one split (compare.average_precision)."""
    p = split_photos(cls, split)
    gts = {i: boxes[boxes["photo_id"] == i][["x", "y", "w", "h"]].values.tolist() for i in p["photo_id"]}
    return cp.average_precision(gts, [(i, s, b) for i in p["photo_id"] for s, b in found[i]], 0.5)

rt_counts, rt_lines = detector_counts(RT_BOXES)
print("score lines chosen on validation:", rt_lines)

## 6. Detector 2: Faster R-CNN, fine-tuned on 67 photos

The same recipe as Module 4: the COCO detector, a new last layer for our two classes (window, car), the backbone frozen, flips, gradient clipping. It learns from the 67 training photos and their 652 boxes.

> **Predict.** RT-DETR already knows cars. Will a detector fine-tuned on only 28 car photos count cars better or worse?

This cell takes about 173 seconds on a laptop CPU.

In [ ]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import to_tensor
EPOCHS = 10  # one epoch is one pass over the training photos
LABEL = {"window": 1, "car": 2}  # 0 is the background
torch.manual_seed(0)
frcnn = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1,
                                          min_size=480, max_size=640, box_detections_per_img=300,
                                          box_score_thresh=0.05)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(frcnn.roi_heads.box_predictor.cls_score.in_features, 3)
for p in frcnn.backbone.body.parameters():
    p.requires_grad = False  # the backbone stays as it is (frozen), as in Module 4
params = [p for p in frcnn.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
train = photos[photos["split"] == "train"]
data = []
for i, f, cls in train[["photo_id", "file", "class"]].itertuples(index=False):
    b = boxes[boxes["photo_id"] == i]
    xyxy = torch.tensor(np.c_[b["x"], b["y"], b["x"] + b["w"], b["y"] + b["h"]], dtype=torch.float32)
    data.append((to_tensor(Image.open("data/crowd/" + f).convert("RGB")),
                 {"boxes": xyxy, "labels": torch.full((len(b),), LABEL[cls], dtype=torch.int64)}))
g = torch.Generator().manual_seed(0)
start = time.perf_counter()
for epoch in range(EPOCHS):
    frcnn.train()
    total = 0.0
    for k in torch.randperm(len(data), generator=g).tolist():
        x, target = data[k]
        if torch.rand(1, generator=g).item() < 0.5:  # flip left to right
            x = x.flip(-1)
            bb = target["boxes"].clone()
            bb[:, [0, 2]] = x.shape[-1] - target["boxes"][:, [2, 0]]
            target = {"boxes": bb, "labels": target["labels"]}
        loss = sum(frcnn([x], [target]).values())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        opt.step()
        total += float(loss)
    print(f"epoch {epoch + 1}: loss {total / len(data):.3f}")
train_seconds = time.perf_counter() - start
print(f"trained on {len(data)} photos in {train_seconds:.0f} s")

In [ ]:
frcnn.eval()
FT_BOXES = {}
start = time.perf_counter()
for i, f, cls in photos[photos["split"] != "train"][["photo_id", "file", "class"]].itertuples(index=False):
    with torch.no_grad():
        o = frcnn([to_tensor(Image.open("data/crowd/" + f).convert("RGB"))])[0]
    FT_BOXES[i] = [(s, [b[0], b[1], b[2] - b[0], b[3] - b[1]])
                   for b, s, l in zip(o["boxes"].tolist(), o["scores"].tolist(), o["labels"].tolist())
                   if l == LABEL[cls]]
ft_ms = 1000 * (time.perf_counter() - start) / len(FT_BOXES)
ft_counts, ft_lines = detector_counts(FT_BOXES)
print(f"{ft_ms:.0f} ms a photo; score lines chosen on validation:", ft_lines)

## 7. A density estimate

The density estimate of [Counting, and tracking in brief](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/): ResNet-18 gives 256 numbers for each 16 x 16 pixel cell; a ridge regression, one per class, learns the number of box centres in each cell from the training photos. The count is the sum over the cells. This cell downloads ResNet-18 (47 MB) and takes about 10 seconds.

In [ ]:
import timm
from huggingface_hub import hf_hub_download
from sklearn.linear_model import Ridge
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision, as in Modules 4 and 6.
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
backbone = timm.create_model("resnet18.a1_in1k", pretrained=True, features_only=True, out_indices=(3,),
                             pretrained_cfg_overlay={"file": weights}).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def cells(path):
    """256 numbers for each 16 x 16 pixel cell of the photo (ResNet-18, layer 3)."""
    x = torch.from_numpy(np.asarray(Image.open(path).convert("RGB"), np.float32) / 255).permute(2, 0, 1)
    with torch.no_grad():
        f = backbone(((x - MEAN) / STD)[None])[0][0]
    return f.permute(1, 2, 0).reshape(-1, f.shape[0]).numpy(), tuple(f.shape[1:])

def centres(photo_id, shape):
    """The target of each cell: the number of true box centres in it."""
    grid = np.zeros(shape)
    for x, y, w, h in boxes[boxes["photo_id"] == photo_id][["x", "y", "w", "h"]].values:
        grid[min(int((y + h / 2) // 16), shape[0] - 1), min(int((x + w / 2) // 16), shape[1] - 1)] += 1
    return grid.ravel()

CELLS = {i: cells("data/crowd/" + f) for i, f in zip(photos["photo_id"], photos["file"])}
density = {}
for cls in CLASSES:
    tr = photos[(photos["class"] == cls) & (photos["split"] == "train")]["photo_id"]
    model = Ridge(alpha=10.0).fit(np.vstack([CELLS[i][0] for i in tr]),
                                  np.concatenate([centres(i, CELLS[i][1]) for i in tr]))
    for i in photos[photos["class"] == cls]["photo_id"]:
        density[i] = float(model.predict(CELLS[i][0]).sum())  # the count is the sum over all cells

## 8. The hosted vision-language model, recorded

The course recorded the answers of a hosted model (Azure `chat-strong`) to one question per photo, so you need no key and no account. No photo with a person in it was sent. The recording budget ran out before the last 19 validation window photos: the model has no answer for them.

In [ ]:
# Recorded answers of a hosted vision-language model (Azure chat-strong), one per photo, replayed here.
vlm_records = [json.loads(line) for line in open("data/crowd/vlm-counts.jsonl")]
vlm = {r["photo_id"]: r["parsed"]["count"] for r in vlm_records}
print(pd.crosstab(pd.Series([r["class"] for r in vlm_records], name="class"),
                  pd.Series([r["split"] for r in vlm_records], name="split")))
print("the question for a window photo:")
print(vlm_records[0]["prompt"] if vlm_records[0]["class"] == "window" else vlm_records[-1]["prompt"])

## 9. A fair comparison

Every counter now has a count for every test photo. The count error (MAE) is the mean of the absolute differences from the true counts; the bias is the mean of the differences (positive: too many). The last three columns are the count error by crowding band.

In [ ]:
METHODS = {"baseline: mean count": baseline, "RT-DETR (COCO, no training)": rt_counts,
           "Faster R-CNN fine-tuned": ft_counts, "density estimate": density, "hosted VLM (recorded)": vlm}
rows = [r for name, counts in METHODS.items() for r in count_rows(name, counts)]
table = pd.DataFrame([{"method": r["method"], "class": r["class"], "count error": r["value"],
                       "bias": r["detail"]["bias"], "exact": r["detail"]["exact"],
                       **{b: r["detail"]["by_crowding"][b]["mae"] for b in BANDS}} for r in rows])
print(table.to_string(index=False))

> **Check.** RT-DETR's count error on cars should be about 1.94, the lowest for cars. On windows it counts nothing: COCO has no window.

The rows also go into the course's comparison table, with the `count` task:

In [ ]:
table = cp.Table()
for r in rows:
    table.add(r | {"setting": f"Open Images {r['class']}s test"})
table.show(task="count", columns=("method", "setting", "value", "worst", "worst_value"))
print("AP50 on the test photos:", {f"{name}, {c}s": ap50(found, c)
                                   for name, found in (("RT-DETR", RT_BOXES), ("fine-tuned", FT_BOXES)) for c in CLASSES})

## 10. The decision: a line to a person, chosen on validation

For each counter and class, `choose_line` finds the cheapest line on validation; then the test photos are opened once. A counter is chosen for each class by its **validation** cost. A counter needs a count for every validation photo to be chosen.

In [ ]:
# what the method itself costs per 1,000 photos: the hosted model's recorded bill; the local models run on the
# office laptop (electricity not counted)
CALLS = {name: 0.0 for name in METHODS}
CALLS["hosted VLM (recorded)"] = round(1000 * np.mean([r["usd"] for r in vlm_records]), 2)
decision = []
for name, counts in METHODS.items():
    for cls in CLASSES:
        line = choose_line(counts, cls)
        val_cost, _ = quote_cost(counts, cls, "val", line)
        no_line, _ = quote_cost(counts, cls, "test")
        cost, checks = quote_cost(counts, cls, "test", line)
        decision.append({"method": name, "class": cls, "line": line, "validation cost": val_cost + CALLS[name],
                         "test cost, no person": no_line + CALLS[name], "test cost with the line": cost + CALLS[name],
                         "photos to a person per 1,000": checks})
decision = pd.DataFrame(decision)
print(decision.to_string(index=False))
# the choice, per class: the method with the lowest validation cost (the test numbers are not used to choose).
# A method must have a count for every validation photo: the recorded model has none for 19 window photos.
complete = {name: {cls: split_photos(cls, "val")["photo_id"].isin(list(counts)).all() for cls in CLASSES}
            for name, counts in METHODS.items()}
chosen = {cls: decision[(decision["class"] == cls) & decision["method"].map(lambda m: complete[m][cls])]
          .sort_values(["validation cost", "method"]).iloc[0]["method"] for cls in CLASSES}
print("chosen on validation:", chosen)

> **Check.** The choice on validation: the fine-tuned detector for windows and the hosted model for cars. Test cost per 1,000 photos: windows 14,458 dollars, cars 9,407.

## 11. End to end: new photos

The decision meets 21 new photos. Each one is quoted from the chosen counter's count, or goes to a person.

In [ ]:
use = {"window": ft_counts, "car": vlm}
for cls in CLASSES:
    line = choose_line(use[cls], cls)
    new = split_photos(cls, "new")
    c = np.array([use[cls][i] for i in new["photo_id"]])
    print(f"{cls}: line {line}, count of each photo:", np.round(c).astype(int).tolist())
    print(f"{' ' * len(cls)}  true counts:            ", new["count"].tolist())
    print(f"{' ' * len(cls)}  cost per 1,000 photos and photos to a person per 1,000:", quote_cost(use[cls], cls, "new", line))

## 12. Your turn

> **Your turn 1.** RT-DETR found no windows. Compute its count error (MAE) on the window test photos in `rt_window_error`, rounded to two decimals. Predict first: what number must it be, without any code?

In [ ]:
test_windows = split_photos("window", "test")
rt_window_error = round(float(np.mean([abs(rt_counts[i] - n) for i, n in zip(test_windows["photo_id"], test_windows["count"])])), 2)
print(rt_window_error)

In [ ]:
expect('rt_window_error', rt_window_error, 17.58, tolerance=0.001)

> **Your turn 2.** The hosted model's count error on windows is large. Is it too high or too low? Compute its **bias** on the window test photos (the mean of its count minus the true count) in `vlm_window_bias`, rounded to two decimals. Then look at two photos where it says far more than the boxes, and decide who is right.

In [ ]:
test_windows = split_photos("window", "test")
vlm_window_bias = round(float(np.mean([vlm[i] - n for i, n in zip(test_windows["photo_id"], test_windows["count"])])), 2)
print(vlm_window_bias)

In [ ]:
expect('vlm_window_bias', vlm_window_bias, 20.89, tolerance=0.001)

> **Your turn 3.** Change one thing: the office hires a second person and can count 500 photos in 1,000. Make a copy of `COSTS` with `checks_per_1000=500`, choose the fine-tuned detector's window line again on validation, and put its window test cost in `budget_500`. Predict first: does the cost fall by half, by less, or by more?

In [ ]:
costs_500 = dict(COSTS, checks_per_1000=500)
line_500 = choose_line(ft_counts, "window", costs_500)
budget_500 = quote_cost(ft_counts, "window", "test", line_500, costs_500)[0]
print(line_500, budget_500)

In [ ]:
expect('budget_500', budget_500, 4833.3, tolerance=0.001)

## Next

The [case study page](https://learning.nextia-ai.com/courses/vision/m09/count-and-detect-in-crowded-scenes/) explains each result, the decision for Clearline, the limits, responsible use, and how to make this a portfolio piece. It also has the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/vision/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m09/count-and-detect-in-crowded-scenes/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are a sample of Open Images V7: each photo by its Flickr author (CC BY 2.0, listed in crowd/manifest.csv), boxes by Google (CC BY 4.0), resized by Nextia Learning; no photo shows a person.